# 子代理 Q 值尺度体检（不改 high_level.py）

本 Notebook 用于在**不修改** `MacroHFT/RL/agent/high_level.py` 的前提下，独立加载 6 个低层子代理（slope_1~3、vol_1~3），在同一批 state 上计算它们输出的 Q 值统计（mean/std/min/max），用于判断是否存在 **Q 值尺度不一致** 导致高层 `mem` 难下降/门控塌缩的问题。

你只需要在第 2 个 cell 里把 `ROOT` 和 `model_list_*` 改成你当前训练产物的路径即可。

In [10]:
import os
import sys
import pathlib
import numpy as np
import pandas as pd
import torch

# ====== 0) 设备选择：为避免 CUDA OOM，默认用 CPU（你也可以手动改回 cuda:0） ======
FORCE_CPU = True
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""  # 让 torch 看不到 GPU，最稳

# ====== 1) 请确认 ROOT 指向 newauthor1009/20260106（即包含 MacroHFT/ 的目录）======
ROOT = "/home/ustc1958/lxy/newauthor1009/20260106"

# 让 Notebook 能 import MacroHFT 工程代码
sys.path.append(ROOT)
sys.path.insert(0, ROOT)
os.chdir(os.path.join(ROOT, "MacroHFT"))

# ====== 2) 导入网络定义与动作空间 ======
from model.net import subagent  # 低层子代理网络
from env.actions import get_actions

# 注意：该 Notebook 的目标是做“尺度体检”，CPU forward 足够且更稳
DEVICE = "cpu" if FORCE_CPU else ("cuda:0" if torch.cuda.is_available() else "cpu")
EPS_DEVICE = DEVICE  # 子代理 forward 的 device

if (not FORCE_CPU) and torch.cuda.is_available():
    torch.cuda.empty_cache()

# 与 high_level.py 一致：ETHUSDT 的 max_holding_number 等不需要在这里用到
tech_indicator_list = np.load('./data/feature_list/single_features.npy', allow_pickle=True).tolist()
tech_indicator_list_trend = np.load('./data/feature_list/trend_features.npy', allow_pickle=True).tolist()

n_state_1 = len(tech_indicator_list)
n_state_2 = len(tech_indicator_list_trend)

# 这里 action_mode/action_size 要与训练子代理时一致
ACTION_MODE = "long"
ACTION_SIZE = 1
ACTIONS, N_ACTION, _ = get_actions(ACTION_MODE, ACTION_SIZE)

LOW_LEVEL_HIDDEN_DIM = 64 * 4


def load_subagent(path: str) -> torch.nn.Module:
    m = subagent(n_state_1, n_state_2, N_ACTION, LOW_LEVEL_HIDDEN_DIM).to(EPS_DEVICE)
    sd = torch.load(path, map_location=EPS_DEVICE)
    m.load_state_dict(sd)
    m.eval()
    return m


def summarize_q(q: torch.Tensor) -> dict:
    # q: [B, n_action]
    q = q.detach().float().cpu()
    return {
        "mean": float(q.mean().item()),
        "std": float(q.std().item()),
        "min": float(q.min().item()),
        "max": float(q.max().item()),
        "p05": float(torch.quantile(q.flatten(), 0.05).item()),
        "p50": float(torch.quantile(q.flatten(), 0.50).item()),
        "p95": float(torch.quantile(q.flatten(), 0.95).item()),
    }


def batch_forward(model: torch.nn.Module, s1: np.ndarray, s2: np.ndarray, prev_a: np.ndarray) -> torch.Tensor:
    """s1/s2: [B, n_state]; prev_a: [B] or [B,1] int.

    关键点：`model.net.subagent.forward()` 里做的是 `embedding(previous_action)`，
    这里 embedding 需要 index tensor 形状为 [B]（而不是 [B,1]）。
    否则 embedding 输出会变成 [B,1,H]，再与 [B,H] 的 trend_state_hidden 相加就会 size mismatch。
    """
    x1 = torch.as_tensor(s1, dtype=torch.float32, device=EPS_DEVICE)
    x2 = torch.as_tensor(s2, dtype=torch.float32, device=EPS_DEVICE)

    pa = torch.as_tensor(prev_a, dtype=torch.long, device=EPS_DEVICE)
    if pa.ndim == 2 and pa.shape[1] == 1:
        pa = pa.view(-1)  # [B]
    elif pa.ndim != 1:
        pa = pa.reshape(-1)

    with torch.no_grad():
        q = model(x1, x2, pa)  # pa: [B]
    return q


def print_table(rows):
    df = pd.DataFrame(rows)
    display(df)
    return df


In [16]:
# ====== 3) 配置：把 6 个子代理 checkpoint 路径改成你当前训练要用的那一套 ======
# 说明：这里直接填 .pth/.pt/.pkl 权重文件路径（torch.load 能读到 state_dict）。

# 你指定要用的 long_action_base（与 high_level.py 里一致）
model_paths = {
    "slope_1": "./result/low_level/ETHUSDT/long_action_base/slope/label_1/1/best_model/best_model.pkl",
    "slope_2": "./result/low_level/ETHUSDT/long_action_base/slope/label_2/4/best_model/best_model.pkl",
    "slope_3": "./result/low_level/ETHUSDT/long_action_base/slope/label_3/0/best_model/best_model.pkl",
    "vol_1": "./result/low_level/ETHUSDT/long_action_base/vol/label_1/4/best_model/best_model.pkl",
    "vol_2": "./result/low_level/ETHUSDT/long_action_base/vol/label_2/1/best_model/best_model.pkl",
    "vol_3": "./result/low_level/ETHUSDT/long_action_base/vol/label_3/1/best_model/best_model.pkl",
}

# ====== 4) 采样同一批 state（无需跑 env，直接造随机特征向量即可做“尺度体检”）======
# 为了避免内存问题，默认 batch 不要太大；并且 forward 时分块跑。
BATCH = 1024
CHUNK = 256
SEED = 0
rng = np.random.default_rng(SEED)

# 用标准正态造特征；尺度体检对分布不敏感（只要所有子代理同一批输入）
s1 = rng.standard_normal((BATCH, n_state_1), dtype=np.float32)
s2 = rng.standard_normal((BATCH, n_state_2), dtype=np.float32)

# subagent.embedding 需要 previous_action 是 [B] 的 index
prev_a = rng.integers(low=0, high=N_ACTION, size=(BATCH,), dtype=np.int64)


def batch_forward_chunked(model: torch.nn.Module, s1: np.ndarray, s2: np.ndarray, prev_a_1d: np.ndarray, chunk: int) -> torch.Tensor:
    """prev_a_1d: [B] int64"""
    outs = []
    for i in range(0, len(prev_a_1d), chunk):
        outs.append(batch_forward(model, s1[i:i+chunk], s2[i:i+chunk], prev_a_1d[i:i+chunk]).cpu())
    return torch.cat(outs, dim=0)


# ====== 5) 加载并统计 ======
rows = []
for name, p in model_paths.items():
    if not os.path.exists(p):
        rows.append({"agent": name, "path": p, "error": "NOT_FOUND"})
        continue
    try:
        m = load_subagent(p)
        q = batch_forward_chunked(m, s1, s2, prev_a, CHUNK)
        stat = summarize_q(q)
        rows.append({"agent": name, "path": p, **stat})

        # 主动释放，避免 notebook 长时间运行占用内存
        del m
        if DEVICE.startswith("cuda") and torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        rows.append({"agent": name, "path": p, "error": repr(e)})

_df = print_table(rows)

# ====== 6) 一个简单的“数量级报警” ======
# 例如：max(mean)/min(mean) > 5 或 max(std)/min(std) > 5
if "mean" in _df.columns:
    means = _df["mean"].dropna()
else:
    means = pd.Series(dtype=float)

if "std" in _df.columns:
    stds = _df["std"].dropna()
else:
    stds = pd.Series(dtype=float)

if len(means) >= 2:
    print("mean_ratio(max/min)=", float(means.max() / max(means.min(), 1e-8)))
if len(stds) >= 2:
    print("std_ratio(max/min)=", float(stds.max() / max(stds.min(), 1e-8)))


,agent,path,mean,std,min,max,p05,p50,p95
0,slope_1,./result/low_level/ETHUSDT/long_action_base/sl...,1.929701,3.303136,-11.553344,12.556728,-3.388338,2.001549,7.185574
1,slope_2,./result/low_level/ETHUSDT/long_action_base/sl...,2.925083,5.375266,-17.738922,11.748818,-7.973192,4.198560,9.494137
2,slope_3,./result/low_level/ETHUSDT/long_action_base/sl...,0.110890,0.630694,-2.965762,3.254179,-0.859280,0.104908,1.134653
3,vol_1,./result/low_level/ETHUSDT/long_action_base/vo...,-1.536400,24.722660,-59.120384,32.421982,-45.370262,8.101426,26.542549
4,vol_2,./result/low_level/ETHUSDT/long_action_base/vo...,-6.997982,19.291391,-65.941696,19.929729,-47.519413,3.187082,9.827530
5,vol_3,./result/low_level/ETHUSDT/long_action_base/vo...,1.696749,2.542119,-7.405581,7.929497,-2.720042,1.951122,5.589779


mean_ratio(max/min)= 292508268.35632324
std_ratio(max/min)= 39.19915654848693


In [28]:
# ====== 3) 配置：把 6 个子代理 checkpoint 路径改成你当前训练要用的那一套 ======
# 说明：这里直接填 .pth/.pt/.pkl 权重文件路径（torch.load 能读到 state_dict）。

# 你指定要用的 long_action_201test high_level.py 里一致）
model_paths = {
    "slope_1": "./result/low_level/ETHUSDT/long_action_201test/slope/label_1/1/best_model/best_model.pkl",
    "slope_2": "./result/low_level/ETHUSDT/long_action_201test/slope/label_2/4/best_model/best_model.pkl",
    "slope_3": "./result/low_level/ETHUSDT/long_action_201test/slope/label_3/0/best_model/best_model.pkl",
    "vol_1": "./result/low_level/ETHUSDT/long_action_201test/vol/label_1/4/best_model/best_model.pkl",
    "vol_2": "./result/low_level/ETHUSDT/long_action_201test/vol/label_2/1/best_model/best_model.pkl",
    "vol_3": "./result/low_level/ETHUSDT/long_action_201test/vol/label_3/1/best_model/best_model.pkl",
}

# ====== 4) 采样同一批 state（无需跑 env，直接造随机特征向量即可做“尺度体检”）======
# 为了避免内存问题，默认 batch 不要太大；并且 forward 时分块跑。
BATCH = 1024
CHUNK = 256
SEED = 0
rng = np.random.default_rng(SEED)

# 用标准正态造特征；尺度体检对分布不敏感（只要所有子代理同一批输入）
s1 = rng.standard_normal((BATCH, n_state_1), dtype=np.float32)
s2 = rng.standard_normal((BATCH, n_state_2), dtype=np.float32)

# subagent.embedding 需要 previous_action 是 [B] 的 index
prev_a = rng.integers(low=0, high=N_ACTION, size=(BATCH,), dtype=np.int64)


def batch_forward_chunked(model: torch.nn.Module, s1: np.ndarray, s2: np.ndarray, prev_a_1d: np.ndarray, chunk: int) -> torch.Tensor:
    """prev_a_1d: [B] int64"""
    outs = []
    for i in range(0, len(prev_a_1d), chunk):
        outs.append(batch_forward(model, s1[i:i+chunk], s2[i:i+chunk], prev_a_1d[i:i+chunk]).cpu())
    return torch.cat(outs, dim=0)


# ====== 5) 加载并统计 ======
rows = []
for name, p in model_paths.items():
    if not os.path.exists(p):
        rows.append({"agent": name, "path": p, "error": "NOT_FOUND"})
        continue
    try:
        m = load_subagent(p)
        q = batch_forward_chunked(m, s1, s2, prev_a, CHUNK)
        stat = summarize_q(q)
        rows.append({"agent": name, "path": p, **stat})

        # 主动释放，避免 notebook 长时间运行占用内存
        del m
        if DEVICE.startswith("cuda") and torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        rows.append({"agent": name, "path": p, "error": repr(e)})

_df = print_table(rows)

# ====== 6) 一个简单的“数量级报警” ======
# 例如：max(mean)/min(mean) > 5 或 max(std)/min(std) > 5
if "mean" in _df.columns:
    means = _df["mean"].dropna()
else:
    means = pd.Series(dtype=float)

if "std" in _df.columns:
    stds = _df["std"].dropna()
else:
    stds = pd.Series(dtype=float)

if len(means) >= 2:
    print("mean_ratio(max/min)=", float(means.max() / max(means.min(), 1e-8)))
if len(stds) >= 2:
    print("std_ratio(max/min)=", float(stds.max() / max(stds.min(), 1e-8)))


,agent,path,mean,std,min,max,p05,p50,p95
0,slope_1,./result/low_level/ETHUSDT/long_action_201test...,-77.179756,140.079346,-429.301941,471.440979,-290.638580,-95.823792,201.533676
1,slope_2,./result/low_level/ETHUSDT/long_action_201test...,-4.157941,7.644120,-48.169830,6.016023,-20.988434,-1.743028,3.388342
2,slope_3,./result/low_level/ETHUSDT/long_action_201test...,-0.020864,1.349811,-5.006827,4.818025,-2.314416,0.107929,1.947508
3,vol_1,./result/low_level/ETHUSDT/long_action_201test...,-37.147820,62.769352,-254.869171,29.831896,-155.036896,-0.141345,19.777975
4,vol_2,./result/low_level/ETHUSDT/long_action_201test...,-0.105469,4.126241,-17.681784,8.526113,-8.256510,0.944596,4.902736
5,vol_3,./result/low_level/ETHUSDT/long_action_201test...,-47.746948,59.566349,-348.850739,108.992119,-168.517227,-36.817490,28.098158


mean_ratio(max/min)= -2086373.046040535
std_ratio(max/min)= 103.77701031569715


In [29]:
# ====== 9) 一键完整验证：Q 尺度 -> 梯度主导 -> 标准化缓解（可重复跑不同目录） ======
# 用法：只需要改 model_paths，然后从本 cell 开始运行即可。

from collections import OrderedDict


def verify_expert_scale_and_grad(model_paths: dict,
                                 s1: np.ndarray,
                                 s2: np.ndarray,
                                 prev_a: np.ndarray,
                                 chunk: int = 256,
                                 seed: int = 0):
    """返回两个 DataFrame：
    1) df_q: 6 个子代理 Q 的统计（mean/std/min/max/分位数）
    2) df_grad_cmp: 标准化前后 mean_abs_dLdw 的对比（越大越主导 w 的训练）

    说明：
    - Q 统计使用真实前向输出，不依赖环境。
    - 梯度主导性分析使用随机上游梯度 up_grad，仅用于量化“尺度如何放大 dL/dw”。
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    # --- 1) forward 取 6 个专家 Q ---
    qs = OrderedDict()
    q_rows = []
    for name, p in model_paths.items():
        m = load_subagent(p)
        q = batch_forward_chunked(m, s1, s2, prev_a, chunk).detach().float()  # [B,A] on CPU
        qs[name] = q
        q_rows.append({
            "agent": name,
            "path": p,
            **summarize_q(q),
        })
        del m

    df_q = pd.DataFrame(q_rows)

    # --- 2) 梯度主导性：dL/dw_i = <up_grad, Q_i> ---
    A = list(qs.values())[0].shape[1]
    B = list(qs.values())[0].shape[0]
    up_grad = torch.randn((B, A))

    grad_rows = []
    grad_rows_norm = []
    for name, q in qs.items():
        dldw = (up_grad * q).sum(dim=1)
        grad_rows.append({
            "agent": name,
            "mean_abs_dLdw": float(dldw.abs().mean().item()),
            "p95_abs_dLdw": float(torch.quantile(dldw.abs(), 0.95).item()),
        })

        # 每个专家按动作维做标准化（与你上面一致）
        qn = (q - q.mean(dim=0, keepdim=True)) / (q.std(dim=0, keepdim=True) + 1e-6)
        dldw_n = (up_grad * qn).sum(dim=1)
        grad_rows_norm.append({
            "agent": name,
            "mean_abs_dLdw_norm": float(dldw_n.abs().mean().item()),
            "p95_abs_dLdw_norm": float(torch.quantile(dldw_n.abs(), 0.95).item()),
        })

    df_grad = pd.DataFrame(grad_rows)
    df_grad_norm = pd.DataFrame(grad_rows_norm)
    df_grad_cmp = df_grad.merge(df_grad_norm, on="agent", how="inner")

    # 排序便于阅读
    df_grad_cmp = df_grad_cmp.sort_values("mean_abs_dLdw", ascending=False)

    return df_q, df_grad_cmp


# -------- 执行验证 --------
df_q, df_grad_cmp = verify_expert_scale_and_grad(
    model_paths=model_paths,
    s1=s1,
    s2=s2,
    prev_a=prev_a,
    chunk=CHUNK,
    seed=0,
)

print("(A) 子代理 Q 分布统计：")
display(df_q[["agent", "mean", "std", "min", "max", "p05", "p50", "p95", "path"]])

print("\n(B) 梯度主导性对比（标准化前 vs 标准化后）：")
display(df_grad_cmp)

# -------- 自动结论 --------
max_raw = float(df_grad_cmp["mean_abs_dLdw"].max())
min_raw = float(df_grad_cmp["mean_abs_dLdw"].min())
max_norm = float(df_grad_cmp["mean_abs_dLdw_norm"].max())
min_norm = float(df_grad_cmp["mean_abs_dLdw_norm"].min())

print("\n(C) 结论：")
print(f"raw_grad_ratio(max/min)  = {max_raw / max(min_raw, 1e-8):.2f}")
print(f"norm_grad_ratio(max/min) = {max_norm / max(min_norm, 1e-8):.2f}")

if max_raw / max(min_raw, 1e-8) > 5 and max_norm / max(min_norm, 1e-8) < 2:
    print("判定：Q 尺度导致 w 梯度严重失衡，是门控塌缩/记忆难对齐的强证据。")
    print("建议：在高层组合前对每个子代理 Q 做标准化/裁剪/温度缩放，再训练高层。")
else:
    print("判定：梯度失衡不显著（或标准化未明显改善），需要进一步检查 env/ reward 语义不一致等因素。")


(A) 子代理 Q 分布统计：


,agent,mean,std,min,max,p05,p50,p95,path
0,slope_1,-77.179756,140.079346,-429.301941,471.440979,-290.638580,-95.823792,201.533676,./result/low_level/ETHUSDT/long_action_201test...
1,slope_2,-4.157941,7.644120,-48.169830,6.016023,-20.988434,-1.743028,3.388342,./result/low_level/ETHUSDT/long_action_201test...
2,slope_3,-0.020864,1.349811,-5.006827,4.818025,-2.314416,0.107929,1.947508,./result/low_level/ETHUSDT/long_action_201test...
3,vol_1,-37.147820,62.769352,-254.869171,29.831896,-155.036896,-0.141345,19.777975,./result/low_level/ETHUSDT/long_action_201test...
4,vol_2,-0.105469,4.126241,-17.681784,8.526113,-8.256510,0.944596,4.902736,./result/low_level/ETHUSDT/long_action_201test...
5,vol_3,-47.746948,59.566349,-348.850739,108.992119,-168.517227,-36.817490,28.098158,./result/low_level/ETHUSDT/long_action_201test...



(B) 梯度主导性对比（标准化前 vs 标准化后）：


,agent,mean_abs_dLdw,p95_abs_dLdw,mean_abs_dLdw_norm,p95_abs_dLdw_norm
0,slope_1,148.237640,477.012329,0.790852,2.750021
5,vol_3,60.776215,213.793182,0.827365,2.873975
3,vol_1,55.767647,236.597412,0.981443,2.826044
1,slope_2,6.189694,24.084126,0.827531,2.495961
4,vol_2,3.665102,12.362910,0.888911,2.924167
2,slope_3,1.242172,3.789892,0.914697,2.893831



(C) 结论：
raw_grad_ratio(max/min)  = 119.34
norm_grad_ratio(max/min) = 1.24
判定：Q 尺度导致 w 梯度严重失衡，是门控塌缩/记忆难对齐的强证据。
建议：在高层组合前对每个子代理 Q 做标准化/裁剪/温度缩放，再训练高层。
